In [1]:
import numpy as np
from numba import njit, prange, types
from numba.typed import List
import pandas as pd
import pickle
import operator

# Book Names

In [2]:
bible_books = [
    "Genesis",
    "Exodus",
    "Leviticus",
    "Numbers",
    "Deuteronomy",
    "Joshua",
    "Judges",
    "Ruth",
    "1 Samuel",
    "2 Samuel",
    "1 Kings",
    "2 Kings",
    "1 Chronicles",
    "2 Chronicles",
    "Ezra",
    "Nehemiah",
    "Esther",
    "Job",
    "Psalms",
    "Proverbs",
    "Ecclesiastes",
    "Song of Solomon",
    "Isaiah",
    "Jeremiah",
    "Lamentations",
    "Ezekiel",
    "Daniel",
    "Hosea",
    "Joel",
    "Amos",
    "Obadiah",
    "Jonah",
    "Micah",
    "Nahum",
    "Habakkuk",
    "Zephaniah",
    "Haggai",
    "Zechariah",
    "Malachi",
    "Matthew",
    "Mark",
    "Luke",
    "John",
    "Acts",
    "Romans",
    "1 Corinthians",
    "2 Corinthians",
    "Galatians",
    "Ephesians",
    "Philippians",
    "Colossians",
    "1 Thessalonians",
    "2 Thessalonians",
    "1 Timothy",
    "2 Timothy",
    "Titus",
    "Philemon",
    "Hebrews",
    "James",
    "1 Peter",
    "2 Peter",
    "1 John",
    "2 John",
    "3 John",
    "Jude",
    "Revelation",
]

# Everything else

In [2]:
ot = pd.read_pickle("pickles/sept.pickle")
nt = pd.read_pickle("pickles/tisch.pickle")
strongs = pd.read_pickle("pickles/strongs.pickle")

In [3]:
bible = pd.concat([ot, nt], axis=0)
all_books = [
    book for _, book in sorted(bible.groupby("book"), key=operator.itemgetter(0))
]
offsets = [0] * len(all_books)
for book in all_books:
    idx = book.iloc[0]["book"]
    if idx == len(offsets):
        print("done")
        break
    offsets[idx] = len(book) + offsets[idx - 1]

done


In [5]:
all_book_strongs = [book["str"].values.astype("<U6") for book in all_books]

In [6]:
reference_type = types.UniTuple(types.int64, 4)
collector_type = types.ListType(reference_type)

@njit(parallel=True)
def look_for_references(all_book_strongs, offsets, bible_books):
    by_referencer = List.empty_list(collector_type)
    for _ in all_book_strongs:
        by_referencer.append(List.empty_list(reference_type))
    for referencer_id in prange(39, len(all_book_strongs)):
        print("looking for references in ", bible_books[referencer_id])
        referencer = all_book_strongs[referencer_id]
        for referenced_id, referenced in enumerate(all_book_strongs[:39]):
            if (
                referencer_id == referenced_id
            ):  # Don't look for references from one book to itself
                continue
            for n in range(5, 6):
                for source_start in range(len(referenced) - n + 1):
                    for quotation_start in range(len(referencer) - n + 1):
                        if (
                            referenced[source_start : source_start + n]
                            == referencer[quotation_start : quotation_start + n]
                        ).all():
                            by_referencer[referencer_id].append(
                                (
                                    quotation_start + offsets[referencer_id],
                                    n,
                                    source_start + offsets[referenced_id],
                                    n,
                                )
                            )
        print("finished finding references in", bible_books[referencer_id])
    found = []
    for subfound in by_referencer:
        found.extend(subfound)
    return found

In [ ]:
found = look_for_references(all_book_strongs, offsets, bible_books)

In [8]:
with open('references-naive.pickle', 'wb') as f:
    pickle.dump(found, f)

In [7]:
for src_ref, src_len, dest_ref, dest_len in found:
    if src_len:
        print(*bible.iloc[src_ref : src_ref + src_len]["text"])
        print(*bible.iloc[dest_ref : dest_ref + dest_len]["text"])
        print()

NameError: name 'found' is not defined